# ROGII Wellbore Geology Prediction Submission

Run this notebook on Kaggle with the competition data attached. It reads `/kaggle/input/rogii-wellbore-geology-prediction/` and writes `/kaggle/working/submission.csv`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd


SEED = 2026
TARGET = "TVT"
SUBMISSION_TARGET = "tvt"
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
PREFERRED_INPUT_DIR = KAGGLE_INPUT_ROOT / "rogii-wellbore-geology-prediction"
OUTPUT_PATH = Path("/kaggle/working/submission.csv")


def well_id_from_path(path: Path) -> str:
    return path.name.split("__", 1)[0].split(".", 1)[0]


def find_input_dir() -> Path:
    candidates: list[Path] = []
    if (PREFERRED_INPUT_DIR / "sample_submission.csv").exists():
        candidates.append(PREFERRED_INPUT_DIR)
    if KAGGLE_INPUT_ROOT.exists():
        candidates.extend(path.parent for path in sorted(KAGGLE_INPUT_ROOT.rglob("sample_submission.csv")))

    unique_candidates = list(dict.fromkeys(candidates))
    valid_candidates = [path for path in unique_candidates if (path / "train").is_dir() and (path / "test").is_dir()]
    if len(valid_candidates) == 1:
        return valid_candidates[0]
    if len(valid_candidates) > 1:
        names = [str(path) for path in valid_candidates]
        raise ValueError(f"Multiple possible competition input directories found: {names}")

    available = sorted(str(path) for path in KAGGLE_INPUT_ROOT.glob("*")) if KAGGLE_INPUT_ROOT.exists() else []
    nested_samples = sorted(str(path) for path in KAGGLE_INPUT_ROOT.rglob("sample_submission.csv")) if KAGGLE_INPUT_ROOT.exists() else []
    raise FileNotFoundError(
        "Could not find a Kaggle input directory containing sample_submission.csv, train/, and test/. "
        f"Available /kaggle/input entries: {available}. "
        f"Nested sample_submission.csv files found: {nested_samples}"
    )


INPUT_DIR = find_input_dir()
print(f"Using competition input directory: {INPUT_DIR}")


def list_files(split: str, kind: str) -> list[Path]:
    folder = INPUT_DIR / split
    if kind == "horizontal":
        pattern = "*__horizontal_well.csv"
    elif kind == "typewell":
        pattern = "*__typewell.csv"
    else:
        raise ValueError(f"Unknown file kind: {kind}")

    files = sorted(folder.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No {kind} files found under {folder}")
    return files


def load_horizontal(split: str) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for path in list_files(split, "horizontal"):
        df = pd.read_csv(path)
        df.insert(0, "well_id", well_id_from_path(path))
        df.insert(1, "row_id", np.arange(len(df), dtype=np.int32))
        frames.append(df)
    return pd.concat(frames, ignore_index=True, sort=False)


def load_typewell_summary(split: str) -> pd.DataFrame:
    rows: list[dict[str, float | int | str]] = []
    for path in list_files(split, "typewell"):
        df = pd.read_csv(path)
        numeric = df.select_dtypes(include=[np.number])
        row: dict[str, float | int | str] = {
            "well_id": well_id_from_path(path),
            "typewell_rows": int(len(df)),
        }
        for col in numeric.columns:
            row[f"typewell_{col}_mean"] = float(numeric[col].mean())
            row[f"typewell_{col}_std"] = float(numeric[col].std())
            row[f"typewell_{col}_min"] = float(numeric[col].min())
            row[f"typewell_{col}_max"] = float(numeric[col].max())
        rows.append(row)
    return pd.DataFrame(rows)


def build_prediction_ids(df: pd.DataFrame) -> pd.Series:
    return df["well_id"].astype(str) + "_" + df["row_id"].astype(str)


def add_features(df: pd.DataFrame, split: str) -> pd.DataFrame:
    work = df.copy()
    work = work.sort_values(["well_id", "row_id"]).reset_index(drop=True)

    type_summary = load_typewell_summary(split)
    if not type_summary.empty:
        work = work.merge(type_summary, on="well_id", how="left")

    base_numeric = [col for col in ["MD", "X", "Y", "Z", "GR"] if col in work.columns]
    for col in base_numeric:
        work[f"{col}_missing"] = work[col].isna().astype(np.int8)
        work[col] = work.groupby("well_id", sort=False)[col].transform(lambda s: s.ffill().bfill())
        work[col] = work[col].fillna(work[col].median())
        work[f"{col}_from_start"] = work[col] - work.groupby("well_id", sort=False)[col].transform("first")
        work[f"{col}_diff1"] = work.groupby("well_id", sort=False)[col].diff().fillna(0.0)

    size = work.groupby("well_id", sort=False)["row_id"].transform("max").replace(0, 1)
    work["row_frac"] = work["row_id"] / size
    work["rows_in_well"] = size + 1

    if "TVT_input" in work.columns:
        work["tvt_input_missing"] = work["TVT_input"].isna().astype(np.int8)
        if "MD" in work.columns:
            known_md = work["MD"].where(work["TVT_input"].notna())
            last_known_md = known_md.groupby(work["well_id"], sort=False).ffill()
            work["md_since_last_tvt_input"] = (work["MD"] - last_known_md).fillna(0.0)
        else:
            work["md_since_last_tvt_input"] = 0.0
    else:
        work["tvt_input_missing"] = 0
        work["md_since_last_tvt_input"] = 0.0

    if "GR" in work.columns:
        group = work.groupby("well_id", sort=False)["GR"]
        work["GR_lag1"] = group.shift(1)
        work["GR_lag3"] = group.shift(3)
        work["GR_roll5_mean"] = group.transform(lambda s: s.rolling(5, min_periods=1).mean())
        work["GR_roll15_mean"] = group.transform(lambda s: s.rolling(15, min_periods=1).mean())
        work["GR_roll15_std"] = group.transform(lambda s: s.rolling(15, min_periods=2).std())
        gr_median = work["GR"].median()
        for col in ["GR_lag1", "GR_lag3", "GR_roll15_std"]:
            work[col] = work.groupby("well_id", sort=False)[col].transform(lambda s: s.bfill().ffill())
            work[col] = work[col].fillna(gr_median)

    return work


def select_common_numeric_features(train: pd.DataFrame, test: pd.DataFrame) -> list[str]:
    excluded = {TARGET, "TVT_input", "well_id", "id"}
    allowed_raw = {"row_id", "MD", "X", "Y", "Z", "GR"}
    allowed_named = {"row_frac", "rows_in_well", "tvt_input_missing", "md_since_last_tvt_input"}
    allowed_prefixes = ("typewell_", "GR_")
    allowed_suffixes = ("_missing", "_from_start", "_diff1")
    common = [col for col in train.columns if col in test.columns]
    features = [
        col
        for col in common
        if col not in excluded and pd.api.types.is_numeric_dtype(train[col]) and pd.api.types.is_numeric_dtype(test[col])
        and (
            col in allowed_raw
            or col in allowed_named
            or col.startswith(allowed_prefixes)
            or col.endswith(allowed_suffixes)
        )
    ]
    if not features:
        raise ValueError("No common numeric features are available for training and prediction.")
    return features


def clean_feature_matrix(train: pd.DataFrame, test: pd.DataFrame, features: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_x = train[features].replace([np.inf, -np.inf], np.nan)
    test_x = test[features].replace([np.inf, -np.inf], np.nan)
    medians = train_x.median(numeric_only=True).fillna(0.0)
    train_x = train_x.fillna(medians)
    test_x = test_x.fillna(medians)
    test_x = test_x.fillna(0.0)
    return train_x, test_x


def build_model() -> tuple[str, object]:
    try:
        from lightgbm import LGBMRegressor

        return "lightgbm", LGBMRegressor(
            objective="regression",
            n_estimators=700,
            learning_rate=0.04,
            num_leaves=63,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=SEED,
            n_jobs=-1,
            verbose=-1,
        )
    except Exception as lightgbm_error:
        try:
            from sklearn.ensemble import HistGradientBoostingRegressor

            return "hist_gradient_boosting", HistGradientBoostingRegressor(
                max_iter=500,
                learning_rate=0.04,
                l2_regularization=0.01,
                random_state=SEED,
            )
        except Exception as hist_error:
            from sklearn.ensemble import RandomForestRegressor

            print(f"LightGBM unavailable: {lightgbm_error}")
            print(f"HistGradientBoosting unavailable: {hist_error}")
            return "random_forest", RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=SEED,
                n_jobs=-1,
            )


def validate_submission(output_path: Path, sample: pd.DataFrame) -> None:
    if not output_path.exists():
        raise FileNotFoundError(f"Submission file was not written: {output_path}")

    sub = pd.read_csv(output_path)
    problems: list[str] = []
    if list(sub.columns) != list(sample.columns):
        problems.append(f"columns {list(sub.columns)} do not match {list(sample.columns)}")
    if SUBMISSION_TARGET not in sub.columns:
        problems.append(f"prediction column must be named {SUBMISSION_TARGET!r}")
    if len(sub) != len(sample):
        problems.append(f"row count {len(sub)} does not match {len(sample)}")
    if "id" in sub.columns and not sub["id"].equals(sample["id"]):
        problems.append("id order does not match sample_submission.csv")
    if SUBMISSION_TARGET in sub.columns:
        values = sub[SUBMISSION_TARGET].to_numpy(dtype=float)
        if not np.isfinite(values).all():
            problems.append("submission contains NaN or infinite predictions")

    if problems:
        raise ValueError(f"Submission validation failed: {problems}")


sample_path = INPUT_DIR / "sample_submission.csv"
if not sample_path.exists():
    raise FileNotFoundError(f"Missing sample submission: {sample_path}")

sample = pd.read_csv(sample_path)
if list(sample.columns) != ["id", SUBMISSION_TARGET]:
    raise ValueError(f"Expected sample submission columns ['id', '{SUBMISSION_TARGET}'], got {list(sample.columns)}")

raw_train = load_horizontal("train")
raw_test = load_horizontal("test")
train = add_features(raw_train, "train")
test = add_features(raw_test, "test")

if TARGET not in train.columns:
    raise ValueError(f"Training data does not contain target column {TARGET!r}")
train = train.loc[train[TARGET].notna()].copy()
if train.empty:
    raise ValueError("No non-missing TVT targets are available for training.")

test["id"] = build_prediction_ids(test)
features = select_common_numeric_features(train, test)
train_x, test_x = clean_feature_matrix(train, test, features)
y = train[TARGET].to_numpy(dtype=float)

model_name, model = build_model()
model.fit(train_x, y)
test_predictions = np.asarray(model.predict(test_x), dtype=float)
if not np.isfinite(test_predictions).all():
    raise ValueError("Model produced non-finite test predictions.")

pred = pd.DataFrame({"id": test["id"].to_numpy(), SUBMISSION_TARGET: test_predictions})
submission = sample[["id"]].merge(pred, on="id", how="left", validate="one_to_one")
if submission[SUBMISSION_TARGET].isna().any():
    missing_ids = submission.loc[submission[SUBMISSION_TARGET].isna(), "id"].head(10).tolist()
    raise ValueError(f"Could not generate predictions for all sample IDs. First missing IDs: {missing_ids}")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission = submission[list(sample.columns)]
submission.to_csv(OUTPUT_PATH, index=False)
validate_submission(OUTPUT_PATH, sample)

print("Kaggle Notebook submission completed.")
print(f"Model: {model_name}")
print(f"Train rows: {len(train)}")
print(f"Test rows: {len(test)}")
print(f"Features: {len(features)}")
print(f"Submission rows: {len(submission)}")
print(f"Wrote: {OUTPUT_PATH}")
print("Validation passed: columns, row count, id order, and finite tvt predictions.")